In [1]:
import pandas as pd
import numpy as np


import pyodbc
from sqlalchemy import create_engine
import urllib.parse
import adodbapi

In [2]:
# Выгрузка данных из куба

def request_to_df(results, cursor): 
    return pd.DataFrame([tuple(row) for row in results], columns=[column[0] for column in cursor.description])

conn_olap_str = "Provider=MSOLAP.8;Data Source=gerbera.korf.int;Initial Catalog=AxOlap_Prod;"

query_obj = '''
SELECT 
    {
	[Measures].[Amount - Deals]
	}
	ON COLUMNS, 
NON EMPTY  
	({[Building Object].[Building Object Id].[Building Object Id].Members} 
    - {[Building Object].[Building Object Id].[Building Object Id].&[_НД]})
    * [Fin Dimension].[Project Name].[Project Name]
	* [Sales].[Sales Id].[Sales Id]	  
	* [Date Deals].[Date From].[Date From]
ON ROWS
FROM [Sales and Deals]
WHERE 
(   [_Currency].[Name].&[0],
	[_Use Cost Correction].[Value].&[0],
	[_Use Cost Type].[Value].&[0],
	[_Use Vat].[Value].&[0],
	[Sales Share].[Sales Share ID].&[2],
	{[Invent Table].[Item Code].Members} 
    - {[Invent Table].[Item Code].&[128702]}
    - {[Invent Table].[Item Code].&[238339]}
    - {[Invent Table].[Item Code].&[604067]}
    - {[Invent Table].[Item Code].&[607866]}
    - {[Invent Table].[Item Code].&[708551]}
    - {[Invent Table].[Item Code].&[773678]}
	)'''
conn_olap = adodbapi.connect(conn_olap_str, timeout=120)
cursor_olap = conn_olap.cursor()
cursor_olap.execute(query_obj)
results = cursor_olap.fetchall()
df_mdx = request_to_df(results, cursor_olap)
cursor_olap.close()
conn_olap.close()

df_mdx = df_mdx.rename(columns={'[Building Object].[Building Object Id].[Building Object Id].[MEMBER_CAPTION]':'Код объекта', 
                                '[Date Deals].[Date From].[Date From].[MEMBER_CAPTION]':'Дата сделки', 
                                '[Measures].[Amount - Deals]':'Сделки - Выручка гросс',
                                '[Sales].[Sales Id].[Sales Id].[MEMBER_CAPTION]':'Код заказа',
                                '[Fin Dimension].[Project Name].[Project Name].[MEMBER_CAPTION]':'Торговый дом'})
df_mdx

# преобразование и группируем данные 
df_mdx_deals = df_mdx.groupby(['Код объекта', 'Торговый дом']).agg(
    **{'Дата сделки': ('Дата сделки', 'min'),  
    'Сделки - Выручка гросс': ('Сделки - Выручка гросс', 'sum')}    
).reset_index()

df_mdx_deals['Торговый дом'] = df_mdx_deals['Торговый дом'].replace('ОВИК', 'НЕД')
df_mdx_deals['Дата сделки'] = pd.to_datetime(df_mdx_deals['Дата сделки'], dayfirst=True)

df_mdx_deals

,Код объекта,Торговый дом,Дата сделки,Сделки - Выручка гросс
0,11433,НЕД,2020-03-13,20000
1,11838,НЕД,2025-12-02,976459.5
2,12965,НЕД,2022-11-17,103337
3,13955,НЕД,2020-01-14,646223.86
4,14294,НЕД,2021-10-21,2341104.5
...,...,...,...,...
25174,ОБ013299,НЕД,2021-01-19,7169893
25175,ОБ013310,НЕД,2019-07-16,738931.50
25176,ОБ013321,НЕД,2020-02-14,463504.72
25177,ОБ013343,НЕД,2019-04-23,3791713.27


In [3]:
# выгрузка данных из БД

# Параметры подключения
server = '172.17.0.19'  
database = 'GPP13'

try:
    # создаем SQLAlchemy engine с Windows Authentication
    params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
    )
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    
    # Запрос к БД ГПП для выгрузки данных по объектам
    query = """with t as (SELECT 
    CASE WHEN b.DATAAREAID = 'ovk' THEN 'НЕД' WHEN b.DATAAREAID = 'krf' THEN 'КОРФ' ELSE 'Другое'     END AS 'Торговый дом',
    CASE WHEN b.DATAAREAID = 'ovk' THEN COALESCE(e2.DIMENSION3_, e1.DIMENSION3_) ELSE e1.DIMENSION3_  END AS 'Филиал',
    CASE WHEN b.DATAAREAID = 'ovk' THEN COALESCE(e2.NAME, e1.NAME) ELSE e1.NAME                       END AS 'Менеджер объекта',
    CASE WHEN b.DATAAREAID = 'ovk' THEN COALESCE(man_cat2.manager_category, man_cat1.manager_category) ELSE man_cat1.manager_category END AS 'Категория менеджера',
	b.ACCOUNTNUM AS                'Код объекта',
    b.NAZN                         'Назначение',
	ROUND(bi.REALSUM_FROM_OBJECT, 0) AS 'Сумма',
	f.[Номер воронки] AS                'Воронка',
	f.[Выход из воронки] AS             'Выход из воронки',
	CAST(f.[Дата выхода из стадии П] AS Date) AS 'Дата выхода из стадии П',
    CAST(b.CREATEDDATE AS Date) AS 'Дата ввода',
    b.STATUSOBJECT                 'Статус объекта',
    COALESCE(CAST(bl.Status_object_date_log AS DATE), CAST(mc.Status_object_date_mc AS DATE)) AS 'Дата взятия статуса'
FROM [GPP13].[dbo].[BOT_OVK] b
LEFT JOIN [GPP13].[dbo].[BOT_OVK_INFO] bi ON b.[ACCOUNTNUM] = bi.[ACCOUNTNUM] AND b.[DATAAREAID] = bi.[DATAAREAID]
LEFT JOIN [AXDB4_NEW].[dbo].[EMPLTABLE] e1 ON b.SALESRESPONSIBLE = e1.EMPLID
LEFT JOIN [AXDB4_NEW].[dbo].[SYSCOMPANYUSERINFO] s ON b.SALESRESPONSIBLE = s.USERID
LEFT JOIN [AXDB4_NEW].[dbo].[EMPLTABLE] e2 ON s.EMPLID = e2.EMPLID
LEFT JOIN AXDB4_NEW.dbo._SalesFunnel f ON b.[ACCOUNTNUM] = f.[Код объекта строительства] AND b.[DATAAREAID] = f.[Торговый дом]
LEFT JOIN (SELECT bcl.[OBJECT], bcl.NEW_VALUE, bcl.DATAAREAID, MAX(bcl.MODIFIEDDATE) as Status_object_date_log 
           FROM GPP13.dbo.BOT_CHANGELOG bcl
           WHERE bcl.FIELD = 'STATUSOBJECT'
           GROUP BY bcl.DATAAREAID, bcl.[OBJECT], bcl.NEW_VALUE
          ) as bl ON b.STATUSOBJECT = bl.NEW_VALUE AND b.[DATAAREAID] = bl.DATAAREAID AND b.ACCOUNTNUM = bl.[OBJECT]
LEFT JOIN (SELECT DATAAREAID, ACCOUNTNUM, STATUSOBJECT, MAX(CREATEDDATE) as Status_object_date_mc
           FROM [GPP13].[dbo].[BOT_OVK_ManagementControl]
           GROUP BY DATAAREAID, ACCOUNTNUM, STATUSOBJECT
          ) as mc ON b.DATAAREAID = mc.DATAAREAID AND b.ACCOUNTNUM = mc.ACCOUNTNUM AND b.STATUSOBJECT = mc.STATUSOBJECT
LEFT JOIN (SELECT manager_id, max(category) AS manager_category 
					FROM GPPV2.dbo.manager_category
					WHERE manager_id <> ''
					GROUP BY manager_id) AS man_cat1 ON b.SALESRESPONSIBLE = man_cat1.manager_id
LEFT JOIN (SELECT manager_id, max(category) AS manager_category 
					FROM GPPV2.dbo.manager_category
					WHERE manager_id <> ''
					GROUP BY manager_id) AS man_cat2 ON s.EMPLID = man_cat2.manager_id
WHERE
    b.STATUS <> 'Отменена'
    AND b.DELETED = 0
    AND b.CREATEDDATE >= '2010-01-01'
    AND b.DATAAREAID IN ('ovk','krf')
)
SELECT
    [Торговый дом],
    CASE
        WHEN [Филиал] = 'Алма-аты' THEN 'Казахстан'
        WHEN [Филиал] = 'НВС' THEN 'Новосибирск'
        WHEN [Филиал] = 'РнД' THEN 'Ростов-на-Дону'
        WHEN [Филиал] = 'СПБ' THEN 'Санкт-Петербург'
        WHEN [Филиал] = 'ЮСХ' THEN 'Южно-Сахалинск'
        ELSE [Филиал]
    END AS 'Филиал',
    [Менеджер объекта],
    [Категория менеджера],
	[Код объекта],
	MAX([Сумма]) AS 'Сумма объекта',
    CASE
        WHEN MAX([Сумма]) >= 30000000 THEN 'A'
        WHEN MAX([Сумма]) >= 12000000 THEN 'B'
        WHEN MAX([Сумма]) >= 3000000 THEN 'C'
        ELSE 'D'
    END AS        'Категория объекта',
    [Назначение],
	[Воронка],
	[Выход из воронки],
	[Дата выхода из стадии П],
    [Дата ввода],
    [Статус объекта],
    [Дата взятия статуса],
    CASE WHEN [Статус объекта] in ('Отгружено', 'Счет/Договор оплачен', 'Счет/Договор оплачен (предоплата)', 'Счёт/Договор предоплачен')  THEN 'Сделка'
         WHEN [Статус объекта] in ('Закрыт лидогенерацией', 'Отказ от мелких сделок' , 'Отказ от реализации объекта на этапе инвестпроекта' , 'Проиграно') THEN 'Проиграно'        
         ELSE 'Активные'   END AS 'Активность'
FROM t
GROUP BY
    [Торговый дом],
    [Филиал],
    [Менеджер объекта],
    [Категория менеджера],
	[Код объекта],
    [Назначение],
	[Воронка],
	[Выход из воронки],
	[Дата выхода из стадии П],
    [Дата ввода],
    [Статус объекта],
    [Дата взятия статуса]
    """
    
    # плучаем датафрейм
    df_sql = pd.read_sql(query, engine)
    print(f"Shape: {df_sql.shape}")   
    
except Exception as e:
    print(f"SQLAlchemy connection error: {str(e)}")

#  Выводим датафрейм 
df_sql

# Приведем даты к datetime
df_sql['Дата ввода'] = pd.to_datetime(df_sql['Дата ввода'])
df_sql['Дата выхода из стадии П'] = pd.to_datetime(df_sql['Дата выхода из стадии П'])
df_sql['Дата взятия статуса'] = pd.to_datetime(df_sql['Дата взятия статуса'])

df_data = df_sql[["Торговый дом",	"Код объекта",	"Сумма объекта", "Воронка",	"Выход из воронки",	"Дата ввода", 
                  "Дата выхода из стадии П", "Филиал",	"Категория объекта",	"Назначение",	"Статус объекта",	"Дата взятия статуса"]]
df_data 

Shape: (189990, 15)


,Торговый дом,Код объекта,Сумма объекта,Воронка,Выход из воронки,Дата ввода,Дата выхода из стадии П,Филиал,Категория объекта,Назначение,Статус объекта,Дата взятия статуса
0,КОРФ,BLD023065,1058314.0,1.0,В стадии П,2016-02-16,NaT,None,D,Газо- и нефтедобыча,Проектирование стадия П,2016-02-16
1,КОРФ,BLD037230,578290.0,4.0,Активные,2016-09-27,2016-09-27,None,D,Легкая промышленность,Проектирование стадия Р,2016-09-27
2,КОРФ,BLD038713,283576.0,4.0,Активные,2016-10-17,2016-10-17,None,D,Легкая промышленность,Проектирование стадия Р,2016-10-17
3,КОРФ,BLD046900,772044.0,4.0,Активные,2017-02-27,2017-02-27,None,D,Медицинские,Ожидание финансирования,2017-02-27
4,КОРФ,BLD052179,1056592.0,2.0,Объекты выпущенные по П,2017-05-12,2017-06-07,None,D,Торгово-развлекательные центры,Проведение тендера на ...,2017-06-07
...,...,...,...,...,...,...,...,...,...,...,...,...
189985,НЕД,BLD251247,109738.0,2.0,Объекты выпущенные по П,2024-10-30,2025-11-19,Южно-Сахалинск,D,Административные здания,Проведение тендера на ...,2025-11-19
189986,НЕД,BLD259993,0.0,4.0,Активные,2025-03-31,NaT,Южно-Сахалинск,D,Административные здания,Проектирование стадия П,2025-03-31
189987,НЕД,BLD269230,17706151.0,1.0,В стадии П,2025-09-03,NaT,Южно-Сахалинск,B,Торгово-развлекательные центры,Проектирование стадия П,2025-09-03
189988,НЕД,BLD273959,400649.0,4.0,Активные,2025-11-28,NaT,Южно-Сахалинск,D,Спецобъекты,Ожидание финансирования,2025-11-28


## Получение основного датасета

In [4]:
# Объединение 2 датафреймов
df_all = df_data.merge(df_mdx_deals, on=["Торговый дом", "Код объекта"], how='left')

# Список статусов, считающихся проигрышами
loss_statuses = ["Проиграно",  "Отказ от реализации объекта на этапе инвестпроекта",  "Отказ от мелких сделок",  "Закрыт лидогенерацией"] 

# Создаем поле "Дата проигрыша"
df_all['Дата проигрыша'] = df_all.apply(lambda row: row['Дата взятия статуса'] if row['Статус объекта'] in loss_statuses else pd.NaT, axis=1)

# Удаляем ненужные поля
df_all = df_all.drop(columns=['Дата взятия статуса', 'Статус объекта'])


# Заполняем "Сумма" из "Сделки - Выручка гросс", если есть, иначе из "Сумма объекта"
#df_all['Сумма'] = df_all['Сделки - Выручка гросс'].fillna(df_all['Сумма объекта'])

# преобразование типов и создание поля 'Сумма' 
df_all['Сумма'] = (pd.to_numeric(df_all['Сделки - Выручка гросс'], errors='coerce')  \
                   .fillna(pd.to_numeric(df_all['Сумма объекта'], errors='coerce'))  \
                   .astype('float64'))

df_all['Сделки - Выручка гросс'] = pd.to_numeric(df_all['Сделки - Выручка гросс'], errors='coerce').astype('float64')
df_all['Сумма объекта'] = pd.to_numeric(df_all['Сумма объекта'], errors='coerce').astype('float64')

# Фильтруем оставляем только 1 и 2 воронки
df_all = df_all.query("Воронка == 1 or Воронка == 2")
df_all.head()

,Торговый дом,Код объекта,Сумма объекта,Воронка,Выход из воронки,Дата ввода,Дата выхода из стадии П,Филиал,Категория объекта,Назначение,Дата сделки,Сделки - Выручка гросс,Дата проигрыша,Сумма
0,КОРФ,BLD023065,1058314.0,1.0,В стадии П,2016-02-16,NaT,None,D,Газо- и нефтедобыча,NaT,NaN,NaT,1058314.0
4,КОРФ,BLD052179,1056592.0,2.0,Объекты выпущенные по П,2017-05-12,2017-06-07,None,D,Торгово-развлекательные центры,NaT,NaN,NaT,1056592.0
5,КОРФ,BLD231937,1209148.0,2.0,Проиграно,2024-01-25,2024-01-25,None,D,Пищевая промышленность,NaT,NaN,2024-01-25,1209148.0
62,КОРФ,BLD022584,13939422.0,2.0,Объекты выпущенные по П,2016-02-08,2016-06-14,None,B,Спортивные,NaT,NaN,NaT,13939422.0
65,КОРФ,BLD048323,2306000.0,1.0,В стадии П,2017-03-17,NaT,None,D,Спецобъекты,NaT,NaN,NaT,2306000.0


In [9]:
#df_all.to_excel('data_tab1.xlsx', index=False) 

## Создание таблицы по месяцам

In [6]:
# Функция для 1 воронки

def prepare_monthly_data_v1(df):
    """
    Анализ 1 воронки по месяцам
    1. Выпущенные из П (Дата выхода из П случилась в конкретном месяце)
    2. Проигрыши (случились в конкретном месяце)
    3. Активные (все объекты имеющиеся в этом месяце, с Датой ввода не позже этого месяца)
    """
    # Создаем базовый DataFrame с месяцами
    months_range = pd.date_range('2023-01-01', '2025-12-31', freq='MS')
    result_df = pd.DataFrame({'Месяц': [m.strftime('%Y-%m') for m in months_range]})
    
    # Добавляем начало и конец каждого месяца для фильтрации
    month_starts = pd.to_datetime(result_df['Месяц'] + '-01')
    month_ends = month_starts + pd.offsets.MonthEnd(1)
    
    all_metrics = []
    
    for idx, month_start in enumerate(month_starts):
        month_end = month_ends[idx]
        metrics = {'Месяц': month_start.strftime('%Y-%m')}

        # важный момент: Для 1 воронки мы должны учитывать ТОЛЬКО объекты, которые находятся в 1 воронке на момент анализа.
        # Но для "Выпущенных" нам нужны объекты, которые ВЫШЛИ в этом месяце, то есть они сейчас во 2 воронке
        
        # Фильтр для 1 воронки
        mask_voronka1 = df['Воронка'] == 1
        
        # 1. ВЫПУЩЕННЫЕ ИЗ П в этом месяце
        # Объекты 1 воронки, у которых Дата выхода из П в этом месяце
        # Важно: они уже во 2 воронке в данных
        mask_released = (
            (df['Дата выхода из стадии П'] >= month_start) & 
            (df['Дата выхода из стадии П'] <= month_end)
        )
        
        released_df = df[mask_released]
        metrics['Выпущенные_кол-во'] = len(released_df)
        metrics['Выпущенные_сумма'] = released_df['Сумма'].sum()
        
        # 2. ПРОИГРЫШИ в этом месяце
        # Объекты 1 воронки, у которых Дата проигрыша в этом месяце
        mask_losses = (
            (df['Дата проигрыша'] >= month_start) & 
            (df['Дата проигрыша'] <= month_end) & 
            mask_voronka1
        )
        
        losses_df = df[mask_losses]
        metrics['Проигрыши_кол-во'] = len(losses_df)
        metrics['Проигрыши_сумма'] = losses_df['Сумма'].sum()
        
        # 3. АКТИВНЫЕ объекты на конец месяца
        # Объекты 1 воронки, у которых:
        # - Дата ввода <= конец месяца, НО не раньше 01.01.2019
        # - Нет Даты выхода из П ИЛИ Дата выхода > конца месяца (еще не выпущены)
        # - Нет Даты проигрыша ИЛИ Дата проигрыша > конца месяца (еще не проиграны)
        
        mask_active = (
           # mask_voronka1 &                                                                          # убрала фильтр по воронке 1!
            ((df['Дата ввода'] >= '2019-01-01') & (df['Дата ввода'] <= month_end)) &                  # введены до конца месяца И Дата ввода  > '2019-01-01'
            (df['Дата выхода из стадии П'].isna() | (df['Дата выхода из стадии П'] > month_end)) &    # не выпущены или выпущены позже
            (df['Дата проигрыша'].isna() | (df['Дата проигрыша'] > month_end))                        # не проиграны или проиграны позже
        )
        
        active_df = df[mask_active]
        metrics['Активные_кол-во'] = len(active_df)
        metrics['Активные_сумма'] = active_df['Сумма'].sum()
        
        # 4. введенные объекты в этом месяце (НОВЫЕ)
        mask_new_objects = ((df['Дата ввода'] >= month_start) & (df['Дата ввода'] <= month_end))  # убрала фильтр по 1 воронке! он не нужен
        metrics['Введено_в_мес_кол-во'] = len(df[mask_new_objects])
        metrics['Введено_в_мес_сумма'] = df.loc[mask_new_objects, 'Сумма'].sum()
        
        # 5. Сумма всех объектов за месяц (Активные + Проигранные + Выпущенные)
        metrics['Всего_в_мес_кол-во'] = (metrics['Активные_кол-во'] +  metrics['Проигрыши_кол-во'] + metrics['Выпущенные_кол-во'])
        metrics['Всего_в_мес_сумма'] = ( metrics['Активные_сумма'] + metrics['Проигрыши_сумма'] + metrics['Выпущенные_сумма'])
        
        all_metrics.append(metrics)
    
    # Объединяем результаты
    metrics_df = pd.DataFrame(all_metrics)
    
    # Переименовываем колонки для 1 воронки
    rename_dict = {
        'Выпущенные_кол-во': '1_Выпущено кол-во',
        'Выпущенные_сумма': '1_Выпущено сумма',
        'Проигрыши_кол-во': '1_Проигрыши кол-во',
        'Проигрыши_сумма': '1_Проигрыши сумма',
        'Активные_кол-во': '1_Активные кол-во',
        'Активные_сумма': '1_Активные сумма'
    }
    metrics_df = metrics_df.rename(columns=rename_dict)
    
    return metrics_df

# Применяем для 1 воронки
df_voronka1_report = prepare_monthly_data_v1(df_all)
print("Отчет по 1 воронке:")
df_voronka1_report

Отчет по 1 воронке:


,Месяц,1_Выпущено кол-во,1_Выпущено сумма,1_Проигрыши кол-во,1_Проигрыши сумма,1_Активные кол-во,1_Активные сумма,Введено_в_мес_кол-во,Введено_в_мес_сумма,Всего_в_мес_кол-во,Всего_в_мес_сумма
0,2023-01,497,4.152435e+09,316,3.261764e+09,9477,8.373514e+10,704,7.637745e+09,10290,9.114934e+10
1,2023-02,655,5.513286e+09,2120,1.065327e+10,7540,7.394836e+10,797,5.921078e+09,10315,9.011492e+10
2,2023-03,1021,1.108383e+10,1855,1.680559e+10,5740,5.506743e+10,969,7.193654e+09,8616,8.295686e+10
3,2023-04,546,4.973886e+09,553,4.880309e+09,5573,5.269369e+10,891,6.716998e+09,6672,6.254788e+10
4,2023-05,503,5.418560e+09,175,1.919548e+09,5780,5.099409e+10,863,5.222208e+09,6458,5.833220e+10
5,2023-06,543,6.101624e+09,170,2.882729e+09,6024,4.991595e+10,926,6.455666e+09,6737,5.890031e+10
6,2023-07,706,9.464104e+09,358,5.918605e+09,5954,4.500059e+10,918,7.573355e+09,7018,6.038330e+10
7,2023-08,568,5.123542e+09,212,1.647141e+09,6236,4.557517e+10,1011,6.939645e+09,7016,5.234585e+10
8,2023-09,588,5.786414e+09,146,1.607278e+09,6598,4.701925e+10,1040,7.889181e+09,7332,5.441294e+10
9,2023-10,550,4.855596e+09,188,1.406031e+09,6918,4.865163e+10,993,7.444201e+09,7656,5.491325e+10


In [7]:
#df_voronka1_report.to_excel('метрики_воронка1.xlsx', index=False) 